# Build the br-ac-react-chat images

Run this notebook from the **project folder** (next to `api/`, `ui/` and `cfn/`) on a **SageMaker AI notebook instance** - they have Docker and the AWS CLI.

It will:
1. **Create two ECR repositories** (`br-ac-react-chat-ui`, `br-ac-react-chat-api`) and an **ECR pull-through cache** rule for ECR Public (AWS CLI; only if missing - re-running is safe)
2. **Build the React UI** with npm (`ui/web` -> `ui/web/dist`)
3. Install the Python packages for the target CPU (UI server and API)
4. **Build and push both images.** The Dockerfiles have no `RUN` steps, so nothing is downloaded during the build; the Python base image comes through the pull-through cache over your ECR VPC endpoint
5. *(optional)* Point an existing `br-ac-react-chat` stack at the new images. For the first deploy, run `scripts/deploy.sh` with the values printed in step 4

**Notebook execution role needs:**
- ECR: `GetAuthorizationToken`; on `repository/br-ac-react-chat-ui` and `repository/br-ac-react-chat-api`: `DescribeRepositories`, `CreateRepository`, `TagResource`, `PutLifecyclePolicy`, and push (`BatchCheckLayerAvailability`, `InitiateLayerUpload`, `UploadLayerPart`, `CompleteLayerUpload`, `PutImage`, `BatchGetImage`, `GetDownloadUrlForLayer`, `DescribeImages`); `DescribePullThroughCacheRules`, `CreatePullThroughCacheRule`; on `repository/br-ac-react-chat-public/*`: `CreateRepository`, `BatchImportUpstreamImage`, `BatchGetImage`, `GetDownloadUrlForLayer`
- KMS (if `ECR_KMS_KEY_ARN` is set): `DescribeKey`, `CreateGrant`, `GenerateDataKey`, `Decrypt` on that key
- Step 5 only: `cloudformation:DescribeStacks`, `UpdateStack`, `DescribeStackEvents`, plus `iam:PassRole` on the stack's roles (or a CloudFormation service role)

## 0. Configuration

In [ ]:
import os, sys, json, time, platform
from pathlib import Path
import boto3

PROJECT         = "br-ac-react-chat"  # repository prefix and CloudFormation stack name
REGION          = boto3.Session().region_name or "ca-central-1"
TARGET_ARCH     = "auto"              # "auto" (this notebook's CPU), "arm64" or "amd64"
BUILD_UI        = True                # False = use the prebuilt ui/web/dist that ships with the project
ECR_KMS_KEY_ARN = ""                  # CMK for the repositories; blank = ECR's AWS managed KMS key. Can't be changed later
DEPLOY_STACK    = True                # step 5: if the stack exists, point it at the new images

# Required tags (same values as the stacks)
TAGS = {"AppCatID": "12345", "Description": "React chat container images", "ServiceName": "AgentCore POC",
        "Support-Team": "CLOUD", "Environment": "dev", "Project": PROJECT}

if TARGET_ARCH == "auto":
    TARGET_ARCH = "arm64" if platform.machine() in ("aarch64", "arm64") else "amd64"
ACCOUNT  = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
REGISTRY = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com"
IMAGE_TAG = time.strftime("v%Y%m%d-%H%M%S")          # tags are immutable - a new one each build
ROOT = Path.cwd()

env = {
    "AWS_DEFAULT_REGION": REGION,
    "PROJECT": PROJECT,
    "REGISTRY": REGISTRY,
    "UI_REPO": f"{PROJECT}-ui",
    "API_REPO": f"{PROJECT}-api",
    "UI_IMAGE_URI": f"{REGISTRY}/{PROJECT}-ui:{IMAGE_TAG}",
    "API_IMAGE_URI": f"{REGISTRY}/{PROJECT}-api:{IMAGE_TAG}",
    "CACHE_PREFIX": f"{PROJECT}-public",
    "BASE_IMAGE": f"{REGISTRY}/{PROJECT}-public/docker/library/python:3.12-slim",
    "TARGET_ARCH": TARGET_ARCH,
    "CPU_ARCHITECTURE": {"arm64": "ARM64", "amd64": "X86_64"}[TARGET_ARCH],   # the stack's CpuArchitecture
    "WHEEL_PLATFORMS": {"arm64": "manylinux2014_aarch64 manylinux_2_28_aarch64",
                        "amd64": "manylinux2014_x86_64 manylinux_2_28_x86_64"}[TARGET_ARCH],
    "BUILD_UI": "true" if BUILD_UI else "false",
    "ECR_KMS_KEY_ARN": ECR_KMS_KEY_ARN,
    "ECR_TAGS_JSON": json.dumps([{"Key": k, "Value": v} for k, v in TAGS.items()]),
    "PYTHON": sys.executable,
    "ROOT": str(ROOT),
    "BUILD": str(ROOT / "build"),
}
os.environ.update(env)      # the %%bash cells below read these

for f in ("api/main.py", "api/Dockerfile", "api/requirements.txt", "api/run/README.txt",
          "ui/server.py", "ui/Dockerfile", "ui/requirements.txt", "ui/run/README.txt",
          "ui/web/package.json", "ui/web/package-lock.json"):
    assert (ROOT / f).exists(), f"{f} not found - open this notebook from the br-ac-react-chat project folder"
for k in ("AWS_DEFAULT_REGION", "UI_IMAGE_URI", "API_IMAGE_URI", "BASE_IMAGE", "TARGET_ARCH", "CPU_ARCHITECTURE"):
    print(f"{k:18s} {env[k]}")

## 1. Create the ECR repositories and pull-through cache (AWS CLI)

Only creates what's missing, so re-running is safe.

- **Repositories `br-ac-react-chat-ui` and `br-ac-react-chat-api`:** KMS-encrypted (your CMK if `ECR_KMS_KEY_ARN` is set), immutable tags, scan on push, keep the last 10 images, required tags.
- **Pull-through cache rule `br-ac-react-chat-public` -> `public.ecr.aws`:** ECR fetches public images on AWS's side, so the Python base image arrives through your `ecr.dkr` VPC endpoint with no internet access.

In [ ]:
%%bash
set -euo pipefail
aws --version

for REPO in "$UI_REPO" "$API_REPO"; do
  if aws ecr describe-repositories --repository-names "$REPO" >/dev/null 2>&1; then
    echo "Repository $REPO already exists"
  else
    ENCRYPTION="encryptionType=KMS"
    [[ -n "$ECR_KMS_KEY_ARN" ]] && ENCRYPTION="encryptionType=KMS,kmsKey=$ECR_KMS_KEY_ARN"
    aws ecr create-repository \
      --repository-name "$REPO" \
      --image-tag-mutability IMMUTABLE \
      --image-scanning-configuration scanOnPush=true \
      --encryption-configuration "$ENCRYPTION" \
      --tags "$ECR_TAGS_JSON" \
      --query "repository.repositoryUri" --output text
  fi
  aws ecr put-lifecycle-policy --repository-name "$REPO" --query "repositoryName" --output text \
    --lifecycle-policy-text '{"rules":[{"rulePriority":1,"description":"Keep the last 10 images","selection":{"tagStatus":"any","countType":"imageCountMoreThan","countNumber":10},"action":{"type":"expire"}}]}'
done

if aws ecr describe-pull-through-cache-rules --ecr-repository-prefixes "$CACHE_PREFIX" >/dev/null 2>&1; then
  echo "Pull-through cache rule $CACHE_PREFIX already exists"
else
  aws ecr create-pull-through-cache-rule \
    --ecr-repository-prefix "$CACHE_PREFIX" \
    --upstream-registry-url public.ecr.aws \
    --query "ecrRepositoryPrefix" --output text
fi

## 2. Build the React UI

`npm ci` installs exactly the versions in `package-lock.json`, then `npm run build` type-checks and writes `ui/web/dist`. Needs Node.js 20+ - if `node` is missing: `conda install -y -c conda-forge nodejs` (or set `BUILD_UI = False` to use the prebuilt `dist` that ships with the project).

In [ ]:
%%bash
set -euo pipefail
cd "$ROOT/ui/web"
if [[ "$BUILD_UI" == "true" ]]; then
  command -v node >/dev/null || { echo "Node.js not found - conda install -y -c conda-forge nodejs, or set BUILD_UI = False"; exit 1; }
  echo "node $(node --version), npm $(npm --version)"
  npm ci --no-audit --no-fund
  npm run build
else
  echo "BUILD_UI = False - using the prebuilt dist/"
fi
test -f dist/index.html || { echo "ERROR: ui/web/dist/index.html missing"; exit 1; }
ls -la dist dist/assets

## 3. Install the Python packages for the target CPU

In [ ]:
%%bash
set -euo pipefail
docker version --format "Docker server {{.Server.Version}}" || {
  echo "Docker isn't available here - use a SageMaker notebook instance, or enable Docker for your Studio domain"; exit 1; }

PLATFORM_ARGS=""
for p in $WHEEL_PLATFORMS; do PLATFORM_ARGS="$PLATFORM_ARGS --platform $p"; done
WRONG=$([[ "$TARGET_ARCH" == arm64 ]] && echo x86_64 || echo aarch64)

rm -rf "$BUILD"
for APP in ui api; do
  PKG="$BUILD/$APP/packages"
  mkdir -p "$PKG"
  # Binary wheels only, for Python 3.12 on the target CPU (same approach as the agent build)
  "$PYTHON" -m pip install --quiet -r "$ROOT/$APP/requirements.txt" --target "$PKG" \
    --python-version 3.12 --implementation cp --only-binary=:all: --upgrade $PLATFORM_ARGS
  rm -rf "$PKG/bin"                                   # console scripts point at this kernel
  find "$PKG" -name __pycache__ -prune -exec rm -rf {} +
  if find "$PKG" -name "*.so" | grep -q "$WRONG"; then echo "ERROR: $WRONG binaries in $APP packages"; exit 1; fi
  echo "$APP packages OK for $TARGET_ARCH: $(du -sh "$PKG" | cut -f1)"
done

## 4. Build and push both images

The Dockerfiles only copy files in, so an arm64 image can also be built on an x86_64 notebook.

In [ ]:
%%bash
set -euo pipefail
# Build contexts: the packages from step 3 plus each app's files
cp "$ROOT/api/main.py" "$ROOT/api/Dockerfile" "$BUILD/api/"
cp -r "$ROOT/api/run" "$BUILD/api/"
cp "$ROOT/ui/server.py" "$ROOT/ui/Dockerfile" "$BUILD/ui/"
cp -r "$ROOT/ui/run" "$BUILD/ui/"
mkdir -p "$BUILD/ui/web" && cp -r "$ROOT/ui/web/dist" "$BUILD/ui/web/"

aws ecr get-login-password | docker login --username AWS --password-stdin "$REGISTRY"

build() {   # build <context> <image>
  local cmd=(docker build --platform "linux/$TARGET_ARCH" --build-arg "BASE_IMAGE=$BASE_IMAGE" -t "$2" "$1")
  if ! DOCKER_BUILDKIT=1 "${cmd[@]}" 2>&1 | tee /tmp/br-ac-react-chat-build.log; then
    if grep -qi buildx /tmp/br-ac-react-chat-build.log; then
      echo "BuildKit isn't installed here - retrying with the classic builder"
      DOCKER_BUILDKIT=0 "${cmd[@]}"
    else
      echo "docker build failed. If it couldn't pull the base image, check the ecr.api / ecr.dkr / s3 endpoints"
      echo "and the role's CreateRepository + BatchImportUpstreamImage on repository/$CACHE_PREFIX/*"
      return 1
    fi
  fi
  local arch; arch=$(docker image inspect "$2" --format "{{.Architecture}}")
  [[ "$arch" == "$TARGET_ARCH" ]] || { echo "ERROR: $2 is $arch, expected $TARGET_ARCH"; return 1; }
  docker push "$2"
}

build "$BUILD/api" "$API_IMAGE_URI"
build "$BUILD/ui" "$UI_IMAGE_URI"

echo
echo "Pushed. For the stack:"
echo "  UI_IMAGE_URI=$UI_IMAGE_URI"
echo "  API_IMAGE_URI=$API_IMAGE_URI"
echo "  CPU_ARCHITECTURE=$CPU_ARCHITECTURE"

## 5. *(optional)* Point the stack at the new images

**First time:** the stack doesn't exist yet - run `scripts/deploy.sh` with the three values printed above.

**After that:** with `DEPLOY_STACK = True` this updates the existing stack, changing only `UiImageUri`, `ApiImageUri` and `CpuArchitecture` (it keeps the deployed template - after changing the template, use `deploy.sh`). Both services roll onto the new images; a failed rollout rolls back automatically.

In [ ]:
cfn = boto3.client("cloudformation", region_name=REGION)
try:
    stack = cfn.describe_stacks(StackName=PROJECT)["Stacks"][0]
except cfn.exceptions.ClientError:
    stack = None

values = {"UiImageUri": env["UI_IMAGE_URI"], "ApiImageUri": env["API_IMAGE_URI"],
          "CpuArchitecture": env["CPU_ARCHITECTURE"]}
if not stack:
    print(f"Stack {PROJECT} doesn't exist yet. Deploy it with:\n"
          f"  UI_IMAGE_URI={values['UiImageUri']} API_IMAGE_URI={values['ApiImageUri']} "
          f"CPU_ARCHITECTURE={values['CpuArchitecture']} ./scripts/deploy.sh")
elif not DEPLOY_STACK:
    print("Skipped. Update the stack with", values)
else:
    params = [{"ParameterKey": p["ParameterKey"], "ParameterValue": values[p["ParameterKey"]]}
              if p["ParameterKey"] in values else {"ParameterKey": p["ParameterKey"], "UsePreviousValue": True}
              for p in stack["Parameters"]]
    cfn.update_stack(StackName=PROJECT, UsePreviousTemplate=True, Parameters=params,
                     Capabilities=["CAPABILITY_IAM"])
    print(f"Updating {PROJECT} to {IMAGE_TAG} - waiting for both services to become healthy ...")
    try:
        cfn.get_waiter("stack_update_complete").wait(
            StackName=PROJECT, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
    except Exception as e:
        print("Update did not complete:", e)
        for ev in cfn.describe_stack_events(StackName=PROJECT)["StackEvents"][:15]:
            if "FAILED" in ev["ResourceStatus"]:
                print(f"  {ev['LogicalResourceId']}: {ev.get('ResourceStatusReason')}")
        raise
    outs = {o["OutputKey"]: o["OutputValue"]
            for o in cfn.describe_stacks(StackName=PROJECT)["Stacks"][0]["Outputs"]}
    print(f"\nDone. Open {outs['AppUrl']}  (self-signed certificate - expect a browser warning)")

## 6. *(optional)* Free disk space on this notebook

In [ ]:
%%bash
docker rmi "$UI_IMAGE_URI" "$API_IMAGE_URI" || true
rm -rf "$BUILD" "$ROOT/ui/web/node_modules"